### 

In [1]:
import os
import cv2
import json

ROOT_CUT_FOLDER = "D:\\Magistrska\\public\\exercise-cut-videos-to-images"
PUBLIC_NEXTJS_FOLDER = "D:\\Magistrska\\blindoff-magistrska\\fitcode-frontend-next\\public\\exercise-videos"


def extract_frames_fps(video_path: str, target_fps: int) -> str:
    """
    Extract frames from a video at a target FPS and save them as images.

    Args:
        video_path: Path to the input video file.
        target_fps: Frames per second to extract.

    Returns:
        Path to the output folder containing the extracted frames.

    Raises:
        FileNotFoundError: If the video cannot be opened.
        RuntimeError: If the FPS cannot be read from the video.
    """
    file_name = os.path.basename(video_path)
    name, _ = os.path.splitext(file_name)

    out_dir = os.path.join(ROOT_CUT_FOLDER, f"{name}_frames_{target_fps}fps")
    images_out_dir = os.path.join(out_dir, "images")

    os.makedirs(images_out_dir, exist_ok=True)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Could not open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    if not fps or fps <= 0:
        raise RuntimeError("Couldn't read FPS from video.")

    # Save 10 frames per second => save one frame every N frames
    step = max(int(round(fps / target_fps)), 1)

    frame_idx = 0
    saved = 0

    while True:
        ok, frame = cap.read()
        if not ok:
            break

        if frame_idx % step == 0:
            # optional: convert BGR->RGB if you prefer PIL-style ordering (not needed for cv2.imwrite)
            out_path = os.path.join(images_out_dir, f"frame_{saved:06d}.jpg")
            cv2.imwrite(out_path, frame)
            saved += 1

        frame_idx += 1

    cap.release()
    print(f"FPS={fps:.3f}, step={step} frames, saved={saved} images to: {out_dir}")

    save_file_names_from_folder_to_json(images_out_dir, out_dir)


def save_file_names_from_folder_to_json(images_folder_dir: str, out_dir: str):
    """
    Saves all image names from a folder into a JSON file
    """
    items = os.listdir(images_folder_dir)
    json_path = os.path.join(out_dir, "images.json")

    with open(json_path, "w") as f:
        json.dump(items, f)


dir = r'D:\Magistrska\blindoff-magistrska\fitcode-frontend-next\public\exercise-cut-videos-to-images\test-blindoff-dataset\images'
save_file_names_from_folder_to_json(dir, dir)

# extract_frames_fps(
#     os.path.join(PUBLIC_NEXTJS_FOLDER, "cmj-bb.mp4"),
#     10,
# )